In [1]:
import torch
print("GPU Available:", torch.cuda.is_available())
print("Device Name:", torch.cuda.get_device_name(0))


GPU Available: True
Device Name: NVIDIA GeForce RTX 4090


In [2]:
import os

class Config:
    BASE_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()

    DATASET_DIR = os.path.join(BASE_DIR, "dataset/chunks")
    SPLIT_DIR = os.path.join(BASE_DIR, "dataset/splits")

    MODEL_DIR = os.path.join(BASE_DIR, "models")
    OUTPUT_DIR = os.path.join(BASE_DIR, "output")

    TRAIN_CSV = os.path.join(SPLIT_DIR, "train.csv")
    TEST_CSV = os.path.join(SPLIT_DIR, "test.csv")
    VAL_CSV = os.path.join(SPLIT_DIR, "val.csv")
    METADATA = os.path.join(BASE_DIR, "dataset/metadata.csv")

    SAMPLE_RATE = 16000
    BATCH_SIZE = 4
    EPOCHS = 15
    LR = 2e-5

    MODEL_NAME = "facebook/mms-1b-all"


In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

cfg = Config()

df = pd.read_csv(cfg.METADATA)

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

train_df, temp_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["region"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    random_state=42,
    stratify=temp_df["region"]
)

# save
os.makedirs(cfg.SPLIT_DIR, exist_ok=True)

train_df.to_csv(cfg.TRAIN_CSV, index=False)
val_df.to_csv(cfg.VAL_CSV, index=False)
test_df.to_csv(cfg.TEST_CSV, index=False)

print("Stratified Split Done!")
print("Train:", len(train_df))
print("Val:", len(val_df))
print("Test:", len(test_df))

c:\Users\USER\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
c:\Users\USER\anaconda3\Lib\site-packages\pandas\core\arrays\masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


Stratified Split Done!
Train: 5320
Val: 665
Test: 665


In [4]:
summary = pd.DataFrame({
    "Train": train_df["region"].value_counts(),
    "Validation": val_df["region"].value_counts(),
    "Test": test_df["region"].value_counts()
}).fillna(0).astype(int)

print("\n--- Split Summary ---")
print(summary)



--- Split Summary ---
            Train  Validation  Test
region                             
barishal     1064         133   133
chittagong   1064         133   133
noakhali     1064         133   133
rangpur      1064         133   133
sylhet       1064         133   133


In [5]:
import numpy as np

def add_noise(audio, noise_factor=0.003):
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise


In [6]:
from torch.utils.data import Dataset
import librosa

class Wav2VecDataset(Dataset):
    def __init__(self, csv_file, config, processor, augment=False):
        self.df = pd.read_csv(csv_file)
        self.cfg = config
        self.processor = processor
        self.augment = augment

    def __len__(self):
        return len(self.df)

    def load_audio(self, path):
        audio, sr = librosa.load(path, sr=self.cfg.SAMPLE_RATE)
        audio = np.nan_to_num(audio)
        audio = audio[:self.cfg.SAMPLE_RATE * 10]  # LIMIT TO 10 SECONDS
        return audio

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = os.path.join(self.cfg.BASE_DIR, row["audio_path"])

        if not os.path.exists(path):
            raise FileNotFoundError(path)

        audio = self.load_audio(path)

        if self.augment:
            if np.random.rand() < 0.3:
                audio = add_noise(audio)

        inputs = self.processor(
            audio,
            sampling_rate=self.cfg.SAMPLE_RATE,
            return_tensors="pt",
            padding=True
        )
        input_values = inputs.input_values[0]

        labels = self.processor.tokenizer(
            row["transcript"],
            return_tensors="pt"
        ).input_ids.squeeze()

        return {
            "input_values": input_values,
            "labels": labels,
            "region": row["region"]
        }


In [7]:
def collate_fn(batch):
    input_values = torch.nn.utils.rnn.pad_sequence(
        [b["input_values"] for b in batch],
        batch_first=True
    )
    labels = torch.nn.utils.rnn.pad_sequence(
        [b["labels"] for b in batch],
        batch_first=True,
        padding_value=-100
    )
    regions = [b["region"] for b in batch]

    return {
        "input_values": input_values,
        "labels": labels,
        "region": regions
    }


In [8]:
from transformers import Wav2Vec2Processor, Wav2Vec2ForCTC
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

# ১. প্রসেসর লোড করার সময় টার্গেট ল্যাংগুয়েজ বাংলা ('ben') বলে দেওয়া
processor = Wav2Vec2Processor.from_pretrained(
    cfg.MODEL_NAME, 
    target_lang="ben",
    ignore_mismatched_sizes=True
)

# ২. Wav2Vec2ForCTC মডেল লোড করা এবং টার্গেট ল্যাংগুয়েজ বাংলা সেট করা
model = Wav2Vec2ForCTC.from_pretrained(
    cfg.MODEL_NAME,
    target_lang="ben",
    ignore_mismatched_sizes=True
)

# ৩. টোকেনাইজারকে বাংলার ডিকোডিং লেয়ার সেট করা
processor.tokenizer.set_target_lang("ben")

model.to(device)
print("Device:", device)
print("✅ Facebook MMS-1B Bengali Pipeline successfully loaded without 404 error!")

preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

c:\Users\USER\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--facebook--mms-1b-all. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/96.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.ben.safetensors:   0%|          | 0.00/9.34M [00:00<?, ?B/s]

Device: cuda
✅ Facebook MMS-1B Bengali Pipeline successfully loaded without 404 error!


In [9]:
from torch.utils.data import DataLoader

train_ds = Wav2VecDataset(cfg.TRAIN_CSV, cfg, processor, augment=True)
val_ds = Wav2VecDataset(cfg.VAL_CSV, cfg, processor, augment=False)
test_ds = Wav2VecDataset(cfg.TEST_CSV, cfg, processor, augment=False)

train_loader = DataLoader(
    train_ds, 
    batch_size=cfg.BATCH_SIZE, 
    shuffle=True, 
    num_workers=4,          
    pin_memory=True,        
    collate_fn=collate_fn
)
val_loader = DataLoader(val_ds, batch_size=cfg.BATCH_SIZE, num_workers=0, collate_fn=collate_fn)
test_loader = DataLoader(test_ds, batch_size=cfg.BATCH_SIZE, num_workers=0, collate_fn=collate_fn)


In [10]:

df_check = pd.read_csv(cfg.TRAIN_CSV)
missing = [os.path.join(cfg.BASE_DIR, p) for p in df_check["audio_path"] if not os.path.exists(os.path.join(cfg.BASE_DIR, p))]
print("Missing Files:", len(missing))
if len(missing) > 0:
    print(missing[:10])


Missing Files: 0


In [ ]:
import torch.optim as optim
from tqdm import tqdm
from transformers import get_linear_schedule_with_warmup 
import os

train_losses = []
val_losses = []

optimizer = optim.AdamW(model.parameters(), lr=cfg.LR, weight_decay=0.01)

num_training_steps = len(train_loader) * cfg.EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=500,
    num_training_steps=num_training_steps
)

# === [সংশোধিত লাইন ১] নতুন PyTorch স্ট্যান্ডার্ড অনুযায়ী স্কেলার ===
scaler = torch.amp.GradScaler('cuda') 

best_loss = float("inf")
patience = 5  
patience_counter = 0

os.makedirs(cfg.MODEL_DIR, exist_ok=True)

for epoch in range(cfg.EPOCHS):
    print("🚀 Epoch:", epoch + 1)

    # facebook/mms-1b-all ব্যবহার করলে model.hubert এর জায়গায় model.wav2vec2 হবে
    # আপনার মডেল অনুযায়ী এই অ্যাট্রিবিউটটি রান করবে
    has_hubert = hasattr(model, "hubert")
    encoder_module = model.hubert.feature_extractor if has_hubert else model.wav2vec2.feature_extractor

    if epoch < 3:
        print("Feature Extractor/Encoder Frozen for stabilization.")
        for param in encoder_module.parameters():
            param.requires_grad = False
    else:
        print("Feature Extractor/Encoder Unfrozen.")
        for param in encoder_module.parameters():
            param.requires_grad = True

    model.train()
    total_loss = 0
    progress_bar = tqdm(train_loader)

    for step, batch in enumerate(progress_bar):
        batch = {k: v.to(device) for k, v in batch.items() if k != "region"}

        # === [সংশোধিত লাইন ২] নতুন স্ট্যান্ডার্ড অনুযায়ী autocast ===
        with torch.amp.autocast('cuda'): 
            outputs = model(**batch)
            loss = outputs.loss

        scaler.scale(loss).backward() 
        
        scaler.unscale_(optimizer) 
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) 

        scaler.step(optimizer) 
        scaler.update() 
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        progress_bar.set_postfix({"loss": loss.item()})

    avg_train_loss = total_loss / len(train_loader)
    print("Train Loss:", avg_train_loss)

    # VALIDATION
    model.eval()
    val_loss = 0

    with torch.no_grad():
        
        with torch.amp.autocast('cuda'): 
            for batch in tqdm(val_loader):
                batch = {k: v.to(device) for k, v in batch.items() if k != "region"}
                outputs = model(**batch)
                val_loss += outputs.loss.item()

    avg_val_loss = val_loss / len(val_loader)
    print("Val Loss:", avg_val_loss)

    train_losses.append(avg_train_loss)
    val_losses.append(avg_val_loss)
    
    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        patience_counter = 0
        checkpoint_path = os.path.join(cfg.MODEL_DIR, "best_model.pt")
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': best_loss,
        }, checkpoint_path)
        print(f"Best model saved with Val Loss: {best_loss:.4f}")
    else:
        patience_counter += 1
        print(f"Loss didn't improve. Early stopping counter: {patience_counter}/{patience}")

    if patience_counter >= patience:
        print("Early stopping triggered. Training terminated!")
        break
    
    torch.save(model.state_dict(), os.path.join(cfg.MODEL_DIR, "model_last.pt"))


KeyboardInterrupt



In [ ]:
import matplotlib.pyplot as plt

os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
epochs = range(1, len(train_losses) + 1)

plt.figure(figsize=(8,5))
plt.plot(epochs, train_losses, label="Train Loss")
plt.plot(epochs, val_losses, label="Val Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Hubert Training vs Validation Loss Curve")
plt.legend()
plt.grid()

save_path = os.path.join(cfg.OUTPUT_DIR, "loss_curve.png")
plt.savefig(save_path, dpi=300, bbox_inches='tight') 
print(f"Loss curve graph successfully saved at: {save_path}")
plt.show()


In [ ]:
import evaluate
import re
from collections import defaultdict

# টেস্ট করার আগে বেস্ট চেকপয়েন্ট রিলোড করা
checkpoint_path = os.path.join(cfg.MODEL_DIR, "best_hubert_model.pt")
if os.path.exists(checkpoint_path):
    checkpoint = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    print("Loaded best weights for evaluation.")

wer_metric = evaluate.load("wer")
cer_metric = evaluate.load("cer")

def clean_bengali_text(text):
    text = re.sub(r'[।,;:!?•\'"()\[\]{}—\-_]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


In [ ]:
def evaluate_model(loader):
    model.eval()
    preds = []
    refs = []

    with torch.no_grad():
        for batch in loader:
            inputs = batch["input_values"].to(device)

            logits = model(inputs).logits
            predicted_ids = torch.argmax(logits, dim=-1)

            pred_text = processor.batch_decode(predicted_ids)

            labels = batch["labels"].clone()
            labels[labels == -100] = processor.tokenizer.pad_token_id
            ref_text = processor.batch_decode(labels)

            cleaned_pred_text = [clean_bengali_text(t) for t in pred_text]
            cleaned_ref_text = [clean_bengali_text(t) for t in ref_text]

            preds.extend(cleaned_pred_text)
            refs.extend(cleaned_ref_text)
            
    wer = wer_metric.compute(predictions=preds, references=refs)
    cer = cer_metric.compute(predictions=preds, references=refs)
    
    word_accuracy = max(0, (1 - wer) * 100)
    char_accuracy = max(0, (1 - cer) * 100)

    return wer, cer, word_accuracy, char_accuracy


In [ ]:
wer, cer, w_acc, c_acc = evaluate_model(test_loader)

print("FINAL RESULT")
print("FINAL TEST DATA RESULT")
print(f"Word Error Rate (WER)     : {wer:.4f}")
print(f"Character Error Rate (CER): {cer:.4f}")
print(f"Word Accuracy (WAcc)      : {w_acc:.2f}%") 
print(f"Character Accuracy (CAcc) : {c_acc:.2f}%") 

txt_save_path = os.path.join(cfg.OUTPUT_DIR, "evaluation_results.txt")
with open(txt_save_path, "w", encoding="utf-8") as f:
    f.write("FINAL RESULT\n")
    f.write("FINAL TEST DATA RESULT\n")
    f.write(f"Word Error Rate (WER)     : {wer:.4f}\n")
    f.write(f"Character Error Rate (CER): {cer:.4f}\n")
    f.write(f"Word Accuracy (WAcc)      : {w_acc:.2f}%\n")
    f.write(f"Character Accuracy (CAcc) : {c_acc:.2f}%\n")

print(f"\nEvaluation results successfully saved at: {txt_save_path}")


In [ ]:
# আঞ্চলিক এক্যুরেসি রিপোর্ট জেনারেট করা (Regional Accuracy Report)
from jiwer import wer as jiwer_wer

region_refs = defaultdict(list)
region_preds = defaultdict(list)

model.eval()
with torch.no_grad():
    for batch in test_loader: 
        region = batch["region"] 
        inputs = batch["input_values"].to(device)

        logits = model(inputs).logits
        predicted_ids = torch.argmax(logits, dim=-1)
        preds = processor.batch_decode(predicted_ids)

        labels = batch["labels"].clone()
        labels[labels == -100] = processor.tokenizer.pad_token_id
        refs = processor.batch_decode(labels)

        for r, pred, ref in zip(region, preds, refs):
            region_preds[r].append(clean_bengali_text(pred))
            region_refs[r].append(clean_bengali_text(ref))

regional_txt_path = os.path.join(cfg.OUTPUT_DIR, "regional_accuracy_report.txt")
print("\nREGIONAL ACCURACY REPORT")

with open(regional_txt_path, "w", encoding="utf-8") as f:
    f.write("REGIONAL ACCURACY REPORT\n")
    f.write("=========================\n\n")
    
    for region in region_refs:
        region_wer = jiwer_wer(region_refs[region], region_preds[region])
        region_w_acc = max(0, (1 - region_wer) * 100) 
        
        print(f"{region}:")
        print(f"   - WER      : {region_wer:.4f}")
        print(f"   - Accuracy : {region_w_acc:.2f}%")
        
        f.write(f"{region}:\n")
        f.write(f"   - WER      : {region_wer:.4f}\n")
        f.write(f"   - Accuracy : {region_w_acc:.2f}%\n\n")

print(f"\nRegional accuracy report successfully saved at: {regional_txt_path}")